In [ ]:
import warnings
warnings.filterwarnings('ignore')
import logging
import lightgbm as lgb
import seaborn as sns
# Suppress LightGBM warnings
logging.getLogger('lightgbm').setLevel(logging.ERROR)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning libraries
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import lightgbm as lgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import optuna

import sys
sys.path.append("..")  # Adds the parent directory to the path
from utilities.correlations import *
from utilities.explanation import *
from utilities.for_ploting import *
from utilities.imputation import *
from utilities.modelling import *
from utilities.preprocessing import *
from utilities.cleaning import *

In [ ]:
print("Loading data...")
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('tested.csv')
train = train_df.copy()
test= test_df.copy()

## EDA

In [ ]:
display_basic_info(train_df)

In [ ]:
target = 'Survived'
exclude_cols = ['PassengerId', 'Survived']
plot_all_num_vs_target(train_df, target)

In [ ]:
plot_all_cat_vs_target(train_df, target,exclude_cols=['Name','Ticket','Cabin','set'])

## Combinatorial Feature Analysis

In [ ]:
plot_one_num_two_cat(train_df,'Age','Sex','Survived')

In [ ]:
plot_three_categorical_correlation(train_df, 'Sex', 'Pclass', target)
plt.show()

In [ ]:
train_df['set'] = 'train'
test_df['set'] = 'test'
all_df = pd.concat([train_df, test_df], axis=0)

In [ ]:
#https://github.com/PacktPublishing/Developing-Kaggle-Notebooks/blob/main/Chapter-03/titanic-start-of-a-journey-around-data-world.ipynb

all_df["Family Size"] = all_df["SibSp"] + all_df["Parch"] + 1
all_df["Age Interval"] = 0.0
all_df.loc[ all_df['Age'] <= 16, 'Age Interval']  = 0
all_df.loc[(all_df['Age'] > 16) & (all_df['Age'] <= 32), 'Age Interval'] = 1
all_df.loc[(all_df['Age'] > 32) & (all_df['Age'] <= 48), 'Age Interval'] = 2
all_df.loc[(all_df['Age'] > 48) & (all_df['Age'] <= 64), 'Age Interval'] = 3
all_df.loc[ all_df['Age'] > 64, 'Age Interval'] = 4
all_df['Fare Interval'] = 0.0
all_df.loc[ all_df['Fare'] <= 7.91, 'Fare Interval'] = 0
all_df.loc[(all_df['Fare'] > 7.91) & (all_df['Fare'] <= 14.454), 'Fare Interval'] = 1
all_df.loc[(all_df['Fare'] > 14.454) & (all_df['Fare'] <= 31), 'Fare Interval']   = 2
all_df.loc[ all_df['Fare'] > 31, 'Fare Interval'] = 3
all_df["Sex_Pclass"] = all_df.apply(lambda row: row['Sex'][0].upper() + "_C" + str(row["Pclass"]), axis=1)
def get_deck(text):
    try:
        return text[0]
    except Exception as ex:
        return "Unknown"
    
all_df["Deck"] = all_df["Cabin"].apply(lambda x: get_deck(x))

In [ ]:
sns.boxplot(x='Survived',y="Family Size",data=all_df[all_df['set']=='train'],palette='viridis')

In [ ]:
#https://github.com/PacktPublishing/Developing-Kaggle-Notebooks/blob/main/Chapter-03/titanic-start-of-a-journey-around-data-world.ipynb

def parse_names(row):
    try:
        text = row["Name"]
        split_text = text.split(",")
        family_name = split_text[0]
        next_text = split_text[1]
        split_text = next_text.split(".")
        title = (split_text[0] + ".").lstrip().rstrip()
        next_text = split_text[1]
        if "(" in next_text:
            split_text = next_text.split("(")
            given_name = split_text[0]
            maiden_name = split_text[1].rstrip(")")
            return pd.Series([family_name, title, given_name, maiden_name])
        else:
            given_name = next_text
            return pd.Series([family_name, title, given_name, None])
    except Exception as ex:
        print(f"Exception: {ex}")
    
all_df[["Family Name", "Title", "Given Name", "Maiden Name"]] = all_df.apply(lambda row: parse_names(row), axis=1)

In [ ]:
#https://github.com/PacktPublishing/Developing-Kaggle-Notebooks/blob/main/Chapter-03/titanic-start-of-a-journey-around-data-world.ipynb

from wordcloud import WordCloud, STOPWORDS
stopwords = set(STOPWORDS)

def show_wordcloud(data, mask=None, title=""):
    text = " ".join(t for t in data.dropna())
    stopwords = set(STOPWORDS)
    stopwords.update(["t", "co", "https", "amp", "U", "Comment", "text", "attr", "object"])
    wordcloud = WordCloud(stopwords=stopwords, scale=4, max_font_size=50, max_words=500,mask=mask, background_color="white").generate(text)
    fig = plt.figure(1, figsize=(12, 12))
    plt.axis('off')
    fig.suptitle(title, fontsize=14)
    fig.subplots_adjust(top=2.3)
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.show()    
show_wordcloud(all_df["Family Name"], title="Family Names on Titanic")

In [ ]:
all_df.head()

In [ ]:
all_df.info()

In [ ]:
# Print number of unique values in each column
print("\nUnique values in each column:")
print(all_df.nunique())

In [ ]:
print(all_df.isnull().sum())

In [ ]:
all_df["Age"].fillna(all_df["Age"].median(), inplace=True)
all_df["Maiden Name"].fillna("Unknown", inplace=True)
all_df["Cabin"].fillna("Unknown", inplace=True)

In [ ]:
#Columns with unique values that are more than 200

unique_counts = all_df.nunique()
unique_columns = unique_counts[unique_counts > 200].index.tolist()
dropped_all_df = all_df.drop(columns=unique_columns + ["set", "Survived"])

cat_columns = dropped_all_df.select_dtypes(include=["object"]).columns.tolist()
for col in cat_columns:
    dropped_all_df[col] = dropped_all_df[col].astype('category')

In [ ]:

X_train = dropped_all_df[all_df["set"] == "train"]
X_test = dropped_all_df[all_df["set"] == "test"]
y_train = all_df[all_df["set"] == "train"]["Survived"]
y_test = all_df[all_df["set"] == "test"]["Survived"]





In [ ]:
model = training_binary_classification_with_lgbm(X_train, y_train, split=0.1)


In [ ]:
print(f"test score of LightGBM: {model.score(X_test, y_test)}")

In [ ]:
features = X_train.columns.tolist()
shap_explain_model_on_batch(model, X_train, features,excluded_cat_features=['Cabin','Deck'])

In [ ]:
#! pip install git+https://github.com/milesqli/iffnn.git 

# Explanation With My IFFNN

In [ ]:
import torch
from iffnn import IFFNN


In [ ]:

dummy_all_df = pd.get_dummies(dropped_all_df, columns=cat_columns, drop_first=True)
X_train = dummy_all_df[all_df["set"] == "train"]
X_test = dummy_all_df[all_df["set"] == "test"]
y_train = all_df[all_df["set"] == "train"]["Survived"]
y_test = all_df[all_df["set"] == "test"]["Survived"]



model = IFFNN(
    input_size=len(X_train.columns),
    num_classes=1,
    feature_names=X_train.columns.tolist(),  # Optional
    class_names=['Not Survived', 'Survived'],      # Optional
    hidden_sizes=[226, 113, 113,113,226], #None,            # Use default hidden layers
    device='auto'                 # Use CUDA if available
)


In [ ]:
# Convert bool columns to int
X_train_NN_ = X_train.astype({col: 'int' for col in X_train.select_dtypes(include='bool').columns})

In [ ]:
from sklearn.model_selection import train_test_split
X_train_NN,X_valid_NN,y_train_NN,y_valid_NN = train_test_split(X_train_NN_,y_train,test_size=0.05)

In [ ]:
# Convert bool columns to int
X_test_NN = X_test.astype({col: 'int' for col in X_test.select_dtypes(include='bool').columns})

In [ ]:
train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.tensor(X_train_NN.values, dtype=torch.float32), torch.tensor(y_train_NN.values, dtype=torch.float32)),
    batch_size=64,
    shuffle=True
)
valid_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.tensor(X_valid_NN.values, dtype=torch.float32), torch.tensor(y_valid_NN.values, dtype=torch.float32)),
    batch_size=64,
    shuffle=True
)


In [ ]:
history = model.train_model(
    train_loader=train_loader,
    valid_loader=valid_loader,
    num_epochs=30, # Adjust as needed
    save_path='best_iffnn_model.pth' # Optional: saves the best model based on validation accuracy
)

In [ ]:
explanations = model.explain(
    X_train_NN.values[:5],   # Your batch of input features
    top_n=5,          # Show top 5 features per class
    print_output=True # Print explanations to console
)

In [ ]:

test_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.tensor(X_test_NN.values, dtype=torch.float32), torch.tensor(y_test.values, dtype=torch.int)),
    batch_size=64,
    shuffle=True
)

In [ ]:
res = model.evaluate_model(test_loader)
score = res['test_accuracy']
print("IFFNN Test Score:", score)